# Descriptive Statistics: Getting a Feel for the Data

> https://www.oreilly.com/library/view/scaling-machine-learning/9781098106812/ch04.html#pearson_correlation_matrix

Machine learning is not magic—you will need to understand your data in order to work with it efficiently and effectively. Getting a solid understanding of the data before you start training your algorithms will save you much time and effort down the road. Fortunately, MLlib provides a dedicated library named pyspark.ml.stat that contains all the functionality you need for extracting basic statistics out of the data.

Don’t worry if that sounds intimidating—you don’t need to fully understand statistics to use MLlib, although some level of familiarity will definitely help you in your machine learning journey. Understanding the data using statistics enables us to better decide on which machine learning algorithm to use, identify biases, and estimate the quality of the data—as mentioned previously, if you put garbage in, you get garbage out. Ingesting low-quality data into a machine learning algorithm will result in a low-performing model. As a result, this part is a must!

Having said that, as long as we build conscious assumptions about what the data looks like, what we can accept, and what we cannot, we can conduct much better experiments and have a better idea of what to remove, what to input, and what we can be lenient about. Take into consideration that those assumptions and any data cleansing operations we perform can have big consequences in production, especially if they are aggressive (like dropping all nulls in a large number of rows or imputing too many default values, which screws up the entropy completely). Watch out for mismatches in assumptions made during the exploratory stages about the data input, quality measurements, and what constitutes “bad” or low-quality data.


## Welcome to the Machine Learning Zoo Project!

For learning about MLlib’s statistics functions, we’ll use the Zoo Animal Classification dataset from the Kaggle repository. This dataset, created in 1990, consists of 101 examples of zoo animals described by 16 Boolean-valued attributes capturing various traits. The animals can be classified into seven types: Mammal, Bird, Reptile, Fish, Amphibian, Bug, and Invertebrate.

The first thing you need to do to get a feel for the data to better plan your machine learning journey is calculate the feature statistics. Knowing how the data itself is distributed will provide you with valuable insights to determine which algorithms to select, how to evaluate the model, and overall how much effort you need to invest in the project.

## Setup Spark

In [ ]:
import findspark
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Zoo").getOrCreate()

In [ ]:
spark

### Loading Data
Download data from 
https://www.kaggle.com/datasets/uciml/zoo-animal-classification?resource=download 
and upload to Databricks

In [ ]:
zoo_data_for_statistics = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("../spark/dataset/zoo.csv")

In [ ]:
zoo_data_for_statistics.printSchema()

In [ ]:
display(zoo_data_for_statistics.head(10))

### Descriptive Statistics with Spark Summarizer

A descriptive statistic is a summary statistic that quantitatively describes or summarizes features from a collection of information. MLlib provides us with a dedicated Summarizer object for computing statistical metrics from a specific column. This functionality is part of the MLlib LinearRegression algorithm for building the Line⁠ar​Regression​Summary. When building the Summarizer, we need to specify the desired metrics. Table below lists the functionality available in the Spark API.

| Metric      | Description                                                                                                                               |
|-------------|-------------------------------------------------------------------------------------------------------------------------------------------|
| mean        | Calculates the average value of a given numerical column                                                                                  |
| sum         | Calculates the sum of the numerical column                                                                                                |
| variance    | Calculates the variance of the column (how far the set of numbers in the column are spread out from its mean value, on average)           |
| std         | Calculates the standard deviation of the column (the square root of the variance value), to provide more weight to outliers in the column |
| count       | Calculates the number of items/rows in the dataset                                                                                        |
| numNonZeros | Finds the number of nonzero values in the column                                                                                          |
| max         | Finds the maximum value in the column                                                                                                     |
| min         | Finds the minimum value in the column                                                                                                     |
| normL1      | Calculates the L1 norm (similarity between the numeric values) of the column                                                              |
| normL2      | Calculates the Euclidean norm of the column                                                                                               |

Note
The L1 and L2 (aka Euclidean) norms are tools for calculating the distance between numeric points in an N-dimensional space. They are commonly used as metrics to measure the similarity between data points in fields such as geometry, data mining, and deep learning.

### Creating Features from dataset 
Like the other MLlib functions, the Summarizer.metrics function expects a vector of numeric features as input. You can use MLlib’s Vector​Assem⁠bler function to assemble the vector.

In [ ]:
from pyspark.ml.feature import VectorAssembler

vecAssembler = VectorAssembler(
    inputCols=[
        "feathers",
        "milk",
        "fins",
        "domestic"
    ],
    outputCol="features"
)
vector_df = vecAssembler.transform(zoo_data_for_statistics)
display(vector_df)

In [ ]:
vector_df.show()

In [ ]:
vector_df.select('animal_name','features').show()

vector is a Dataset, that's a strongly typed collection of object.

With dataset is possible to access to columns directly (no need to use []) 

https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Column.html

In [ ]:
vector_df.select(vector_df.features).head(10)

In [ ]:
vector_df.select(vector_df['features']).head(10)

### Compute statistics

In [ ]:
# Import libraries
from pyspark.ml.stat import Summarizer
summarizer = Summarizer.metrics("mean","sum","variance","std")

In [ ]:
summarizer

In [ ]:
# compute statistics for multiple metrics
statistics_df = vector_df.select(summarizer.summary(vector_df.features))
# statistics_df will plot all the metrics
statistics_df.show()

In [ ]:
# compute statistics for single metric (here, std) without the rest
vector_df.select(Summarizer.std(vector_df.features)).show()

The standard deviation (STD) is an indicator of the variation in a set of values. A low STD indicates that the values tend to be close to the mean (also called the expected value) of the set, while a high STD indicates that the values are spread out over a wider range.

Note
Summarizer.std is a global function that you can use without creating a Summarizer instance.

Since the features feathers, milk, fins, and domestic are inherently of type Boolean—milk can be 1 for true or 0 for false, and the same for fins and so on—calculating the STD doesn’t provide us with much insight—the result will always be a decimal number between 0 and 1. That misses the value of STD in calculating how “spread out” the data is. Instead, let’s try the sum function. This function will tell us how many animals in the dataset have feathers, milk, or fins or are domestic animals:

In [ ]:
# compute statistics for single metric "sum" without the rest
vector_df.select(Summarizer.sum(vector_df.features)).show(truncate=False)

This tells us that there are 20 animals with feathers (the vector’s first value), 41 animals that provide milk (the vector’s second value), 17 animals with fins (the third value), and 13 domestic animals (the final value). The sum function provides us with more insights about the data itself than the std function, due to the Boolean nature of the data. However, the more complicated/diverse the dataset is, the more looking at all of the various metrics will help.

## Data Skewness
Skewness in statistics is a measure of the asymmetry of a probability distribution. Think of a bell curve where the data points are not distributed symmetrically on the left and right sides of the curve’s mean value. Assuming the dataset follows a normal distribution curve, skewness means it has a short tail on one end and a long tail on the other. The higher the skewness value is, the less evenly distributed the data is, and the more data points will fall on one side of the bell curve.

To measure skewness, or the asymmetry of the values around the mean, we need to extract the mean value and calculate the standard deviation. A statistical equation to accomplish this has already been implemented in Spark for us; check out the next code snippet to see how to take advantage of it:

In [ ]:
from pyspark.sql.functions import skewness 
df_with_skew =  vector_df.select(skewness("legs")).show()

What does it mean ?

Skewness = 0.137 is very close to 0, meaning:
- The distribution of the number of legs across animals in the dataset is approximately symmetric.
- There is a slight right skew, suggesting there are a few animals with unusually high leg counts (like centipedes or other invertebrates)

## Correlation

*Correlation* computes the correlation matrix for the input Dataset of Vectors using the specified method. 
The output will be a DataFrame that contains the correlation matrix of the column of vectors.


A correlation between two features means that if feature A increases or decreases, feature B does the same (a positive correlation) or does the exact opposite (a negative correlation). Determining correlation therefore involves measuring the linear relationship between the two variables/features. Since a machine learning algorithm’s goal is to learn from data, perfectly correlated features are less likely to provide insights to improve model accuracy. This is why filtering them out can significantly improve our algorithm’s performance while maintaining the quality of the results. The test method of the ChiSquareTest class in MLlib is a statistical test that helps us assess categorical data and labels by running a Pearson correlation on all pairs and outputting a matrix with correlation scores.

Warning
Be mindful that correlation doesn’t necessarily imply causation. When the values of two variables change in a correlated way, there is no guarantee that the change in one variable causes the change in the other. It takes more effort to prove a causative relationship.

>

### Pearson correlation
When looking into correlation, we look for positive or negative associations. Pearson correlation measures the strength of linear association between two variables. It produces a coefficient r that indicates how far away the data points are from a descriptive line. The range of r is [–1,1], where:

- r=1 is a perfect positive correlation. Both variables act in the same way.
- r=–1 is perfect negative/inverse correlation, which means that when one variable increases, the other decreases.
- r=0 means no correlation.

In [ ]:
from pyspark.ml.stat import Correlation

# compute r1 0 Pearson correlation
r1 = Correlation.corr(vector_df, "features").head()
print("Pearson correlation matrix:\n" + str(r1[0])+ "\n")

Each line represents the correlation of a feature with all the other features, in a pairwise way: for example, r1[0][0,1] represents the correlation of feathers with milk, which is a negative value (-0.41076061) that indicates a negative correlation between animals that produce milk and animals with feathers.

|          | feathers    | milk        | fins        | domestic    |
|----------|-------------|-------------|-------------|-------------|
| feathers | 1           | -.41076061  | -0.22354106 | 0.03158624  |
| milk     | -0.41076061 | 1           | -0.15632771 | 0.16392762  |
| fins     | -0.22354106 | -0.15632771 | 1           | -0.09388671 |
| domestic | 0.03158624  | 0.16392762  | -0.09388671 | 1           |

This table makes it easy to spot negative and positive correlations: for example, fins and milk have a negative correlation, while domestic and milk have a positive correlation.

### Spearman correlation
Spearman correlation, also known as Spearman rank correlation, measures the strength and direction of the monotonic relationship between two variables. In contrast to Pearson, which measures the linear relationship, this is a curvilinear relationship, which means the association between the two variables changes as the values change (increase or decrease). Spearman correlation should be used when the data is discrete and the relationships between the data points are not necessarily linear, as shown in Figure 4-4, as well as when ranking is of interest. To decide which approach fits your data better, you need to understand the nature of the data itself: if it’s on an ordinal scale,2 use Spearman, and if it’s on an interval scale,3 use Pearson. To learn more about this, I recommend reading Practical Statistics for Data Scientists by Peter Bruce, Andrew Bruce, and Peter Gedeck (O’Reilly).
![](https://www.oreilly.com/api/v2/epubs/9781098106812/files/assets/smls_0404.png)

In [ ]:
from pyspark.ml.stat import Correlation

# compute r2 0 Spearman correlation
r2 = Correlation.corr(vector_df, "features", "spearman").head()
print("Spearman correlation matrix:\n" + str(r2[0]))

## A dummy example from scratch

In [ ]:
from pyspark.ml.linalg import Vectors
from pyspark.ml.stat import Correlation

# Each row is an observation; each observation has multiple features
data = [
    (Vectors.dense([1.0, 2.0]),),  # Observation 1: A=1.0, B=2.0
    (Vectors.dense([2.0, 4.0]),),  # Observation 2: A=2.0, B=4.0
    (Vectors.dense([3.0, 6.0]),),  # Observation 3: A=3.0, B=6.0
    (Vectors.dense([4.0, 8.0]),)   # Observation 4: A=4.0, B=8.0
]
# the comma after the variable forces Python to consider it as a tuple
# https://www.w3schools.com/python/gloss_python_tuple_one_item.asp
# Credits Ernesto Casablanca, TAP 2021-04-26

df = spark.createDataFrame(data, ["features"])
df.show()

In [ ]:
r1 = Correlation.corr(df, "features").head()
print("Pearson correlation matrix:\n" + str(r1[0]))

r2 = Correlation.corr(df, "features", "spearman").head()
print("Spearman correlation matrix:\n" + str(r2[0]))

## Hypothesis testing
Hypothesis testing is a powerful tool in statistics to determine whether a result is statistically significant, whether this result occurred by chance or not. spark.ml currently supports Pearson’s Chi-squared ( χ2) tests for independence.

ChiSquareTest conducts Pearson’s independence test for every feature against the label. For each feature, the (feature, label) pairs are converted into a contingency matrix for which the Chi-squared statistic is computed. All label and feature values must be categorical.

In [ ]:
from pyspark.ml.linalg import Vectors
from pyspark.ml.stat import ChiSquareTest
from  pyspark.ml.feature import StringIndexer

In [ ]:
stringIndexer = StringIndexer(inputCol="animal_name", outputCol="label")
model = stringIndexer.fit(vector_df)
zoodf = model.transform(vector_df)

In [ ]:
display(zoodf.select('animal_name','label'))

In [ ]:
r = ChiSquareTest.test(zoodf, "features", "label").head()
print("pValues: " + str(r.pValues))
print("degreesOfFreedom: " + str(r.degreesOfFreedom))
print("statistics: " + str(r.statistics))

In [ ]:
str(r.pValues)

## A Dice example

In [ ]:
from pyspark.ml.linalg import Vectors
from pyspark.ml.stat import ChiSquareTest

#https://it.wikipedia.org/wiki/Test_chi_quadrato
data = [(1, Vectors.dense(333,333)),
        (2, Vectors.dense(333,333)),
        (3, Vectors.dense(333,333)),
        (4, Vectors.dense(333,333)),
        (5, Vectors.dense(333,333)),
        (6, Vectors.dense(333 ,333))]
df = spark.createDataFrame(data, ["label", "features"])
df.show()
r = ChiSquareTest.test(df, "features", "label").head()
print("pValues: " + str(r.pValues))
print("degreesOfFreedom: " + str(r.degreesOfFreedom))
print("statistics: " + str(r.statistics))

In [ ]:
spark.stop()